# Gradix: pre-grading visual para cartas Pokémon TCG

Este notebook documenta la construcción del dataset y la lógica analítica principal del proyecto **Gradix**, una aplicación de visión computacional orientada a la **pre-evaluación visual** de cartas Pokémon TCG.

El objetivo del proyecto es transformar capturas fotográficas de cartas en un conjunto estructurado de variables cuantitativas que permitan:

- evaluar la calidad de captura,
- estimar señales visuales de condición,
- construir un dataset reproducible,
- y entrenar un modelo supervisado para clasificar cartas como `damaged` o `undamaged`.

La app representa la capa de producto para el usuario final. En cambio, este notebook representa la capa analítica y reproducible del proyecto.

# Problema y objetivo

En mercados de cartas coleccionables, el valor económico depende en gran medida de la **condición física** del objeto. Sin embargo, evaluar esa condición de forma consistente a partir de una fotografía no es trivial: intervienen factores como blur, iluminación, reflejos, encuadre, geometría, bordes, esquinas y superficie.

El problema abordado en este proyecto consiste en diseñar un pipeline que, a partir de una imagen de una carta Pokémon TCG:

1. detecte y rectifique la carta,
2. extraiga señales visuales relevantes,
3. convierta esas señales en variables tabulares,
4. y permita clasificar operativamente la carta como dañada o no dañada.

El objetivo no es sustituir un servicio profesional de grading, sino construir una herramienta de **pre-evaluación visual operativa**, útil como apoyo para análisis preliminar, organización de inventario y estimación inicial de condición.

# Fuente de datos

El dataset parte de imágenes propias organizadas en una estructura de carpetas del tipo:

```text
data/raw/
  damaged/
  undamaged/


---

## 4) Celda markdown — Pipeline técnico

```markdown
# Pipeline de visión computacional

El pipeline principal de Gradix sigue la siguiente secuencia lógica:

1. **Detección de carta**  
   Se localiza la carta dentro de la imagen mediante un bloque de detección geométrica.

2. **Rectificación (warp)**  
   Una vez detectadas las esquinas, la carta se transforma a una vista frontal más estable para análisis.

3. **Validación post-warp**  
   Se verifica que la carta rectificada conserve una geometría razonable y suficiente calidad para continuar.

4. **Extracción de variables visuales**  
   Se obtienen métricas como blur, brillo, contraste, cobertura, centrado, bordes, esquinas y superficie.

5. **Scoring heurístico**  
   Se generan scores intermedios de captura, centrado, bordes, esquinas y whitening/surface.

6. **Construcción del dataset**  
   Cada imagen se convierte en una fila tabular con variables numéricas, etiquetas, estado técnico de análisis y scores derivados.

7. **Modelado supervisado**  
   Finalmente, el dataset se utiliza para entrenar un clasificador binario de condición.

# Modelado supervisado

Una vez construido el dataset tabular a partir del pipeline visual, se entrenó un modelo supervisado para predecir si una carta pertenece a la clase `damaged` o `undamaged`.

El objetivo de esta etapa no fue reemplazar una certificación profesional de grading, sino evaluar si las variables visuales, geométricas y heurísticas extraídas por Gradix contienen señal suficiente para discriminar entre cartas dañadas y no dañadas dentro del esquema operativo definido para este proyecto.

## Variables utilizadas

La matriz de entrada se construyó a partir de columnas numéricas derivadas del pipeline, agrupadas en los siguientes bloques:

- `visual_`: métricas básicas de la imagen rectificada, como blur, brillo y contraste.
- `geometry_`: cobertura de carta y calidad del aspect ratio.
- `centrado_`: márgenes y balances horizontales/verticales.
- `borde_`: métricas por lado asociadas a bordes.
- `esquina_`: métricas por esquina.
- `superficie_`: whitening, glare, textura y dark spots.
- `score_capture_`, `score_prelim_`, `score_centering_`, `score_edge_`, `score_corner_`, `score_ws_`: scores intermedios del pipeline.

Se excluyeron variables no numéricas y algunas columnas con riesgo de fuga o poco valor operativo directo.

In [ ]:
from pathlib import Path
import json
import joblib
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    balanced_accuracy_score,
)

DATASET_PATH = Path("/content/dataset_gradix.csv")
RANDOM_STATE = 42

def build_feature_matrix(df: pd.DataFrame):
    allowed_prefixes = (
        "visual_",
        "geometry_",
        "centrado_",
        "borde_",
        "esquina_",
        "superficie_",
        "score_capture_",
        "score_prelim_",
        "score_centering_",
        "score_edge_",
        "score_corner_",
        "score_ws_",
    )

    banned_exact = {
        "geometry_used_fallback",
        "ratio_centrado_calculado",
    }

    feature_cols = []
    for col in df.columns:
        if col in banned_exact:
            continue
        if col.startswith(allowed_prefixes):
            feature_cols.append(col)

    X = df[feature_cols].copy()

    bool_cols = X.select_dtypes(include=["bool"]).columns.tolist()
    for col in bool_cols:
        X[col] = X[col].astype("int8")

    non_numeric_cols = X.select_dtypes(exclude=["number"]).columns.tolist()
    if non_numeric_cols:
        X = X.drop(columns=non_numeric_cols)

    X = X.replace([np.inf, -np.inf], np.nan)

    empty_cols = [c for c in X.columns if X[c].isna().all()]
    if empty_cols:
        X = X.drop(columns=empty_cols)

    return X

In [ ]:
df = pd.read_csv(DATASET_PATH)

df = df[df["target_damaged"].isin([0, 1])].copy()
df["target_damaged"] = df["target_damaged"].astype(int)

X = build_feature_matrix(df)
y = df["target_damaged"].copy()

sample_weight = np.where(df["analysis_status"].eq("valid_with_warning"), 1.0, 0.50)

X_train, X_test, y_train, y_test, w_train, w_test = train_test_split(
    X,
    y,
    sample_weight,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y,
)

model = HistGradientBoostingClassifier(
    learning_rate=0.05,
    max_iter=300,
    max_depth=6,
    min_samples_leaf=20,
    l2_regularization=0.1,
    random_state=RANDOM_STATE,
)

model.fit(X_train, y_train, sample_weight=w_train)

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

metrics = {
    "accuracy": float(accuracy_score(y_test, y_pred)),
    "balanced_accuracy": float(balanced_accuracy_score(y_test, y_pred)),
    "precision": float(precision_score(y_test, y_pred, zero_division=0)),
    "recall": float(recall_score(y_test, y_pred, zero_division=0)),
    "f1": float(f1_score(y_test, y_pred, zero_division=0)),
    "confusion_matrix": confusion_matrix(y_test, y_pred).tolist(),
    "n_rows": int(len(df)),
    "n_features": int(X.shape[1]),
}

metrics

In [ ]:
print("=== Classification report ===")
print(classification_report(y_test, y_pred, digits=4, zero_division=0))

cm = confusion_matrix(y_test, y_pred)
cm_df = pd.DataFrame(
    cm,
    index=["Real: undamaged", "Real: damaged"],
    columns=["Pred: undamaged", "Pred: damaged"]
)
cm_df

## Resultados del modelo

El modelo final trabajó con 1,960 observaciones y 133 variables de entrada. Sobre el conjunto de prueba alcanzó un accuracy de 0.908, un balanced accuracy de 0.909, una precisión de 0.936, un recall de 0.903 y un F1-score de 0.919. La matriz de confusión obtenida fue:

- Verdaderos negativos: 151
- Falsos positivos: 14
- Falsos negativos: 22
- Verdaderos positivos: 205

Estos resultados indican que el modelo logra separar de forma consistente ambas clases dentro del marco operativo del proyecto, con un equilibrio razonable entre precisión y sensibilidad.

## Interpretación

Los resultados sugieren que las variables generadas por Gradix sí contienen señal útil para distinguir entre cartas dañadas y no dañadas. Esto es relevante porque valida el enfoque del proyecto: no solo se trata de construir una interfaz visual atractiva, sino de demostrar que el pipeline de visión computacional puede transformarse en un dataset tabular útil para modelado supervisado.

Además, el balanced accuracy cercano a 0.91 sugiere que el modelo conserva estabilidad entre clases, lo cual es importante cuando se busca evitar que el desempeño aparente dependa únicamente de una clase dominante.

## Implementación en la aplicación

El modelo entrenado no se queda únicamente en el notebook. Sus resultados se integran a la app de Gradix como una capa adicional de interpretación: a partir de las métricas visuales y de condición calculadas por el pipeline, la interfaz muestra una predicción binaria de daño, su probabilidad asociada y un nivel de confianza.

De esta forma, el proyecto conecta tres niveles:

1. procesamiento visual de la carta,
2. construcción de variables cuantitativas,
3. inferencia supervisada utilizable dentro de la interfaz final.